# aou_covid — v18.7 re-run

Re-runs the pipeline after the pre-index leakage fix (`223f715`) and adds the
wave x exposure interaction test (`02d`).

**What was wrong.** `01_aou_etl.py` STEP 6 built `num_diagnosis` and
`ehr_length_days` with no index-date restriction, so codes accrued *during* the
COVID admission counted toward the propensity model. `num_diagnosis` is the
strongest matching term (pre-match SMD 0.489), so controls were partly selected
to match a product of the outcome.

**Run the cells in order.** Cell 2 may make cell 7 unnecessary.
Read the output of cell 5 before going further — it is the cell that says whether
the fix actually took effect.


## 0. Configuration


In [ ]:
import os, re, subprocess, textwrap, json, sys
import pandas as pd

COHORT   = 'aou_v7'          # <- 'aou_v8' if this workspace is on a newer CDR
BRANCH   = 'review/v18.7-reconcile'
REPO_URL = 'https://github.com/Su-informatics-lab/aou_covid.git'
REPO_DIR = os.path.expanduser('~/aou_covid')
BUCKET   = os.environ.get('WORKSPACE_BUCKET', '')
BDIR     = f'{BUCKET}/data/covid_sdoh/{COHORT}'

def sh(cmd, check=True):
    print(f'$ {cmd}')
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-8000:])
    if p.returncode and p.stderr: print('STDERR:', p.stderr[-4000:])
    if check and p.returncode: raise RuntimeError(f'failed: {cmd}')
    return p

print('bucket :', BUCKET or '(WORKSPACE_BUCKET is not set -- uploads will be skipped)')
print('cohort :', COHORT)


## 1. Get the code

Public repo, so no credentials. If the branch is already checked out this just
fast-forwards it.


In [ ]:
if not os.path.isdir(REPO_DIR):
    sh(f'git clone {REPO_URL} {REPO_DIR}')
os.chdir(REPO_DIR)
sh('git fetch --all --prune')
sh(f'git checkout {BRANCH}')
sh(f'git reset --hard origin/{BRANCH}')
sh('git log --oneline -3')
os.makedirs(f'results/{COHORT}', exist_ok=True)

# The rename must be complete or the ETL dies on its last line.
src = open('01_aou_etl.py').read()
assert '"survey_ord"' in src and 'enrollment_ord' not in src.split('save(')[-1], \
    'survey_ord rename is not complete in 01_aou_etl.py'
assert 'survey_ord' in open('01b_psm.R').read(), 'survey_ord missing from 01b_psm.R'
print('OK: survey_ord rename is consistent across the ETL and MatchIt')


## 2. Probe the bucket first — this may close eTable 12b/12c without a re-run

`02c` has written its outputs to the workspace bucket since the commit that created
it. The four `wave_stratified_*.csv` were never pulled back into git, which is why
eTable 12b/12c are unverifiable from the repository. If they are still in the bucket,
they can be recovered as-is.

These are products of the **old** matching set. Recovering them closes traceability,
not staleness — they get regenerated in cell 7 either way.


In [ ]:
if BUCKET:
    sh(f'gsutil ls {BDIR}/ | sort', check=False)
    p = sh(f'gsutil ls {BDIR}/wave_stratified_*.csv', check=False)
    if p.returncode == 0 and p.stdout.strip():
        sh(f'gsutil -m cp {BDIR}/wave_stratified_*.csv results/{COHORT}/', check=False)
        sh(f'gsutil -m cp {BDIR}/wave_joint_sdoh_*_coefficients.csv results/{COHORT}/', check=False)
        print('\nRECOVERED. Commit these to git before anything else overwrites them:')
        sh(f'ls -la results/{COHORT}/wave_*', check=False)
    else:
        print('\nNot in the bucket. 02c in cell 7 will regenerate them.')
else:
    print('No WORKSPACE_BUCKET; skipping.')


## 3. Snapshot the OLD matching variables, so the change is measurable

Without a baseline there is no way to tell a working fix from a silent no-op.


In [ ]:
BASE = None
if BUCKET:
    p = sh(f'gsutil cp {BDIR}/06_matching_variables.csv /tmp/06_matching_OLD.csv', check=False)
    if p.returncode == 0:
        BASE = pd.read_csv('/tmp/06_matching_OLD.csv')
        print('old matching variables:', BASE.shape)
        print(BASE[[c for c in ('num_diagnosis','ehr_length_days') if c in BASE]].describe())
if BASE is None:
    print('No baseline available. Cell 5 will fall back to the published values:')
    print('  pre-match SMD for num_diagnosis = 0.489; case median num_diagnosis = 114')


## 4. ETL — the fixed STEP 6

Takes a while; it is BigQuery-bound.


In [ ]:
sh(f'python 01_aou_etl.py {COHORT.replace("aou_","")}')


## 5. STOP AND READ — did the index restriction actually take effect?

The single check this whole re-run exists for.

| what | expect |
|---|---|
| `num_diagnosis` median | clearly **lower** than before; unchanged means the CASE WHEN is not biting |
| `num_diagnosis` == 0 | now appears — people with no pre-index diagnosis |
| `ehr_length_days` NaN | now appears; `dropna` drops these people, so the cohort shrinks |
| cohort size | **record the loss** — Figure 1 and the Table 1 N both have to follow it |


In [ ]:
NEW = pd.read_csv(f'results/{COHORT}/06_matching_variables.csv')
coh = pd.read_csv(f'results/{COHORT}/01_covid_cohort.csv')
print('rows', len(NEW), ' unique person_id', NEW.person_id.nunique())
assert len(NEW) == NEW.person_id.nunique(), 'STEP 6 returned more than one row per person'
assert 'survey_ord' in NEW.columns, NEW.columns.tolist()

print('\n--- new ---'); print(NEW[['num_diagnosis','ehr_length_days']].describe())
if BASE is not None and 'num_diagnosis' in BASE:
    o, n = BASE.num_diagnosis.median(), NEW.num_diagnosis.median()
    print(f'\nnum_diagnosis median  old {o:.0f} -> new {n:.0f}   ({100*(n-o)/o:+.1f}%)')
    if n >= o: print('*** WARNING: did not fall. Check the CASE WHEN in match_sql before continuing. ***')

print('\nzeros in num_diagnosis :', int((NEW.num_diagnosis == 0).sum()))
print('NaN in ehr_length_days :', int(NEW.ehr_length_days.isna().sum()))
cc = NEW.dropna(subset=['survey_ord','num_diagnosis','ehr_length_days'])
print(f'complete matching vars : {len(cc):,} / {len(NEW):,}  (lost {len(NEW)-len(cc):,})')
cases = set(coh.loc[coh.severity == 1, 'person_id'])
print(f'cases retained         : {cc.person_id.isin(cases).sum():,}   (published: 4,064)')
print('\n^ if the case count moved, Figure 1 and every N in the paper move with it.')


## 6. Matching


In [ ]:
sh(f'Rscript 01b_psm.R {COHORT}')
smd = pd.read_csv(f'results/{COHORT}/07c_smd_pre_matching.csv')
print(smd.to_string(index=False))
print('\npublished pre-match SMD for num_diagnosis was 0.489; it should now be smaller.')


## 7. Models, sensitivity, tables


In [ ]:
for cmd in [f'Rscript 02_models.R {COHORT}',
            f'Rscript 02b_variance_sensitivity.R {COHORT}',
            f'Rscript 02c_wave_stratified_race_insurance.R {COHORT}',
            f'python 01c_sensitivity_etl.py {COHORT.replace("aou_","")}',
            f'Rscript 03_sensitivity.R {COHORT}',
            f'python 04_tables.py {COHORT}']:
    sh(cmd)


## 8. NEW — wave x exposure interaction (D11)

Closes the third Introduction question, which the Results currently declines
("not compared formally"). One pooled model per exposure; the primary test is a
person_id-clustered Wald test on the interaction block, matching how every other
estimate in the study is clustered.

Power estimated from the published wave-stratified estimates: **race x wave z ~ 4.5**
(pre-Delta vs Omicron), **income x wave z ~ 1.0**. Expect race to be significant and
income not to be — which is what the paper already claims in words.


In [ ]:
sh(f'Rscript 02d_wave_interaction.R {COHORT}')
print(pd.read_csv(f'results/{COHORT}/wave_interaction_tests.csv').to_string(index=False))


## 9. The numbers that decide what has to change in the manuscript


In [ ]:
R = f'results/{COHORT}'
def show(path, **kw):
    try:
        d = pd.read_csv(path)
        for k, v in kw.items(): d = d[d[k].astype(str).str.contains(v, na=False)]
        print(d.to_string(index=False))
    except Exception as e: print(f'  [{path}: {e}]')

print('=== chronic pulmonary / mild liver: still inverse? ===')
print('If these cross toward 1.0, the Discussion paragraph explaining them as a',
      'normal consequence of encounter-density matching must be deleted --',
      'the leakage was the explanation.')
show(f'{R}/base_model_coefficients.csv', variable='Pulmonary|Liver_Disease_Mild')

print('\n=== profile odds-ratio contrast: 1.78 finally gets an interval ===')
show(f'{R}/profile_odds_ratio_contrast.csv')

print('\n=== joint SDoH ===')
show(f'{R}/joint_sdoh_coefficients.csv', variable='f.insurance|f.income|f.employment|f.housing')

print('\n=== race attenuation ===')
show(f'{R}/race_attenuation_table.csv')


## 10. Push everything back, and say what still has to reach git

The bucket is not the repository. That distinction is exactly how eTable 12b/12c
went missing the first time.


In [ ]:
if BUCKET:
    sh(f'gsutil -m cp {R}/*.csv {BDIR}/', check=False)
    sh(f'gsutil -m cp {R}/*.txt {BDIR}/', check=False)
    print('\nuploaded. now in the bucket:')
    sh(f'gsutil ls {BDIR}/', check=False)

print(textwrap.dedent('''
    NEXT, OFF-PLATFORM
    1. Pull results/ down and `git add` it. Include:
         wave_stratified_*.csv          (closes eTable 12b/12c)
         wave_interaction_*.csv         (closes Introduction question 3)
         profile_odds_ratio_contrast.csv (gives 1.78 its interval)
    2. python 05_figures.py ; python 06_supplement.py ; python make_figures.py
       Do NOT redraw Figure 1 or Figure 2 -- those are the draw.io originals.
    3. bash analysis/gate.sh check
       It will fail widely and that is correct. Update each assertion to the new
       artifact. Never edit a result to satisfy an assertion.
    4. bash analysis/gate.sh ledger
    5. Re-run MarketScan on Quartz (01_ms_etl.py -> 01b -> 02 -> 04).
'''))
